# DataMate · Thử agent trên Colab

Notebook này chạy **một câu hỏi mỗi lần**, lưu JSON và hiện dấu vết SQL. Không có bước training.

1. Theo phần D1–D2 của `docs/huong-dan-chay-tung-buoc.md`, đưa `datamate-colab-source.zip` và `train.zip` vào Drive/MyDrive/DataMate.
2. Chọn GPU trong Runtime → Change runtime type.
3. Chạy lần lượt bốn ô code bên dưới. Ở ô cuối, sửa câu hỏi rồi chạy lại để tiếp tục thử, không cần load lại model.

Chọn model ở `MODEL_KEY` trong ô đầu; `qwen35-9b` là Qwen chính thức. BF16 (`QUANTIZATION="none"`) cần GPU đủ VRAM. Có thể đổi `MODEL_KEY` sang `qwen35-9b` (Qwen chính thức) hoặc `qwen35-upscaled14b-v2` (Pinkstackorg/Qwen-3.5-upscaled-14B-V2-noft). Kiểm tra VRAM của GPU thực tế trước khi load. Lượt Pinkstack cũ có lỗi EOS/parser. Source ngày 06/11 đã sửa giao tiếp chung cho Qwen 9B/14B; chưa có benchmark GPU sau sửa. Các output đang lưu là lịch sử, cần runtime mới để kiểm tra bản này.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, sys, json, zipfile, subprocess, runpy

BASE = Path("/content/drive/MyDrive/DataMate")
SOURCE_ZIP = BASE / "datamate-colab-source.zip"
TRAIN_ZIP = BASE / "train.zip"
WORK = Path("/content/datamate-inference")
SAVE = BASE / "runs/inference-01"
MODEL_KEY = "qwen35-9b"  # "qwen35-9b", "qwen35-upscaled14b-v2", or "gemma4-12b"
QUANTIZATION = "none"    # "none" only when GPU/RAM is sufficient
MODEL_SOURCE_DIR = BASE / "models" / MODEL_KEY
MODEL_DIR = MODEL_SOURCE_DIR  # Ô load sẽ chọn bản cache local

WORK.mkdir(parents=True, exist_ok=True)
SAVE.mkdir(parents=True, exist_ok=True)
def extract_checked(archive, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        if any(not (destination/name).resolve().is_relative_to(destination) for name in z.namelist()):
            raise ValueError("Unsafe ZIP path")
        z.extractall(destination)
extract_checked(SOURCE_ZIP, WORK)
archive_helper = WORK / "tools/colab_archives.py"
if not archive_helper.is_file():
    raise RuntimeError("Gói source cũ. Chạy tools/package_colab.py trên máy, thay datamate-colab-source.zip trên Drive rồi chạy lại ô này.")
archives = runpy.run_path(str(archive_helper))
TRAIN_ROOT = archives["prepare_train_archive"](
    TRAIN_ZIP, WORK, WORK / "configs/training_sources.json"
)
os.chdir(WORK)
os.environ["HF_HOME"] = str(WORK/".cache/huggingface")
os.environ["TORCH_HOME"] = str(WORK/".cache/torch")
os.environ["PIP_CACHE_DIR"] = str(WORK/".cache/pip")
print("Train:", TRAIN_ROOT, "Model:", MODEL_KEY, "Outputs:", SAVE)


Mounted at /content/drive
TRAIN OK: 162 file, fingerprint 5d598c5d603020c8077df7802cc6fb06b364af9914eb88425bd7664c639978ea
Train: /content/datamate-inference/uploaded-train/train-pb2p9s3s/train
Train: /content/datamate-inference/uploaded-train/train-pb2p9s3s/train Model: qwen35-9b Outputs: /content/drive/MyDrive/DataMate/runs/inference-01


## 2. Cài thư viện, kiểm tra GPU
Giữ bản PyTorch CUDA do Colab cung cấp. Nếu resolver báo xung đột thì dừng và kiểm tra, không tự đổi PyTorch.

In [2]:
import torch
constraints = WORK / "colab-constraints.txt"
constraints.write_text("torch==" + torch.__version__ + "\n")
subprocess.run([sys.executable,"-m","pip","install","-e",".[llm]","-c",str(constraints)],cwd=WORK,check=True)
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv"],check=True)
assert torch.cuda.is_available(), "Select a CUDA GPU runtime first"
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

import importlib
source_dir = (WORK / "src").resolve()
if not (source_dir / "datamate/__init__.py").is_file():
    raise RuntimeError("Gói source thiếu src/datamate. Thay source ZIP mới rồi chạy ô đầu.")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()
import datamate
print("DataMate import OK:", datamate.__version__, datamate.__file__)


PyTorch: 2.14.1+cu130 CUDA: 13.0
DataMate import OK: 0.2.0 /content/datamate-inference/src/datamate/__init__.py


## 3. Chuẩn bị và load model từ ổ tạm Colab
Model trên Drive được giữ lâu dài; mỗi runtime chép một lần sang `/content/models`, rồi kiểm tra toàn bộ SHA-256 và nạp tại đó. Dùng lại cache cùng manifest khi chạy lại ô; vẫn kiểm tra hash trước khi nạp. Không đổi model, quantization hoặc cấu hình inference.
Ô hiển thị 3 giai đoạn và tổng thời gian. Nếu đã có agent, chỉ chạy ô hỏi/batch. Ổ tạm cần đủ chỗ cho trọng số; dữ liệu cache không được giữ khi runtime bị xóa.


In [3]:
import sys, importlib
from pathlib import Path

source_dir = (Path(WORK) / "src").resolve()

if not (source_dir / "datamate/__init__.py").is_file():
    raise RuntimeError("Chưa có mã nguồn. Hãy chạy lại ô 1.")

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

importlib.invalidate_caches()

from datamate.config import Settings
print("Đã import DataMate thành công.")

Đã import DataMate thành công.


In [4]:
# This entire file is embedded in the inference notebook's model-loading cell.
from pathlib import Path
import gc, os, runpy, subprocess, sys, time

runtime_helper = Path(WORK) / 'tools/colab_runtime_check.py'
if not runtime_helper.is_file():
    raise RuntimeError('Source ZIP cũ: thay datamate-colab-source.zip mới rồi chạy lại ô chuẩn bị.')
verify_runtime = runpy.run_path(str(runtime_helper))['verify_runtime']
print('Runtime preflight:', verify_runtime(WORK), flush=True)
from datamate.config import Settings
from datamate.application import create_agent
import torch
verify_runtime(WORK)

load_started = time.monotonic()
if 'MODEL_SOURCE_DIR' not in globals():
    MODEL_SOURCE_DIR = Path(MODEL_DIR)  # Compatibility with the earlier setup cell.
MODEL_SOURCE_DIR = Path(MODEL_SOURCE_DIR)
print('[1/3] Kiểm tra bản model lưu lâu dài:', MODEL_SOURCE_DIR, flush=True)
prepare_started = time.monotonic()
prepare_env = {**os.environ, 'HF_HUB_OFFLINE':'0', 'TRANSFORMERS_OFFLINE':'0'}
if not (MODEL_SOURCE_DIR / 'artifact_manifest.json').exists():
    subprocess.run([sys.executable, '-m', 'datamate', 'stage-model', '--model', MODEL_KEY,
                    '--output', str(MODEL_SOURCE_DIR)], env=prepare_env, cwd=WORK, check=True)
print('Chuẩn bị nguồn:', round(time.monotonic()-prepare_started, 2), 'giây', flush=True)

helper_path = WORK / 'tools/colab_model_cache.py'
if not helper_path.is_file():
    raise RuntimeError('Gói source cũ: thay datamate-colab-source.zip mới rồi chạy lại ô chuẩn bị.')
print('[2/3] Chép/dùng lại model trên ổ tạm Colab', flush=True)
prepare_local_model = runpy.run_path(str(helper_path))['prepare_local_model']
MODEL_DIR = prepare_local_model(MODEL_SOURCE_DIR, Path('/content/models'), MODEL_KEY)

if 'agent' in globals():
    del agent
    gc.collect()
    torch.cuda.empty_cache()
settings = Settings(model_key=MODEL_KEY, model_dir=str(MODEL_DIR),
                    mode='self_review',
                    quantization=QUANTIZATION, split_root=str(TRAIN_ROOT),
                    cache_dir=str(WORK/'.cache/datamate'), result_dir=str(SAVE),
                    max_context_tokens=16384, max_new_tokens=768,
                    max_steps=6, question_timeout_seconds=90,
                    schema_mode='code_first', adaptive_preparation=False,
                    require_plan=True, auto_execute_plan=True,
                    schema_repair_timeout_seconds=20, schema_repair_max_new_tokens=512)
print('[3/3] Kiểm tra SHA-256 local một lần, rồi nạp model lên GPU', flush=True)
started = time.monotonic()
agent = create_agent(settings)
print('Kiểm tra + nạp:', round(time.monotonic()-started, 2), 'giây', flush=True)
print('Loaded:', MODEL_KEY, 'tổng thời gian ô:', round(time.monotonic()-load_started, 2), 'giây', flush=True)
print('Pipeline:', agent.pipeline_version, '| mode:', settings.mode, '| schema:', settings.schema_mode, '| auto_execute_plan:', settings.auto_execute_plan, flush=True)
print('Generation tokens:', getattr(agent.backend, 'generation_tokens', None), flush=True)
print('Giữ agent này để hỏi/chạy batch; không cần chạy lại ô load.', flush=True)


Runtime preflight: {'pipeline_version': 6, 'backend_protocol': 3, 'runner_sha256': '85a1b5d2a92e90ba9820a0db0bf3e30c3c90220bf494327706d0540537cfb5fc'}
[1/3] Kiểm tra bản model lưu lâu dài: /content/drive/MyDrive/DataMate/models/qwen35-9b
Chuẩn bị nguồn: 0.41 giây
[2/3] Chép/dùng lại model trên ổ tạm Colab
Cache local: /content/models/qwen35-9b/f8b32a3f0b36609c4065a72af44938fa27ba68e4b2d70ef45a8d43b1cc37399e
Cần chép 16 file, 18.00 GiB
  [1/16] .source.json (0.00 GiB)
  [2/16] LICENSE (0.00 GiB)
  [3/16] README.md (0.00 GiB)
  [4/16] chat_template.jinja (0.00 GiB)
  [5/16] config.json (0.00 GiB)
  [6/16] merges.txt (0.00 GiB)
  [7/16] model.safetensors-00001-of-00004.safetensors (4.91 GiB)
  [8/16] model.safetensors-00002-of-00004.safetensors (4.97 GiB)
  [9/16] model.safetensors-00003-of-00004.safetensors (5.00 GiB)
  [10/16] model.safetensors-00004-of-00004.safetensors (3.10 GiB)
  [11/16] model.safetensors.index.json (0.00 GiB)
  [12/16] preprocessor_config.json (0.00 GiB)
  [13/16] 

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Kiểm tra + nạp: 22.79 giây
Loaded: qwen35-9b tổng thời gian ô: 766.42 giây
Pipeline: 6 | mode: self_review | schema: code_first | auto_execute_plan: True
Generation tokens: {'checkpoint_eos_token_ids': [248044], 'tokenizer_eos_token_id': 248046, 'tokenizer_pad_token_id': 248044, 'chat_turn_token_ids': {'<|im_end|>': 248046}, 'effective_eos_token_ids': [248044, 248046], 'effective_pad_token_id': 248044}
Giữ agent này để hỏi/chạy batch; không cần chạy lại ô load.


## 4. Đặt câu hỏi, xem SQL
Mỗi lần bấm chạy ô này chỉ hỏi một câu. Đáp án chuẩn trong answers.jsonl không được đưa cho agent. Đúng một câu thử không phải benchmark accuracy.

In [5]:
from datamate.domain import Question, json_text
from datetime import datetime, timezone

BUSINESS_ID = "biz_101"
AS_OF_DATE = "2026-02-22"
QUESTION = "Hôm nay cửa hàng bán được bao nhiêu tiền?"
# Then try: "Tổng doanh thu tháng này là bao nhiêu?"
question_id = "interactive_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
run = agent.run(Question(question_id,BUSINESS_ID,QUESTION,AS_OF_DATE))
payload = {"model_key":MODEL_KEY,"business_id":BUSINESS_ID,"as_of_date":AS_OF_DATE,
           "question":QUESTION,**run.to_dict(question_id)}
output = SAVE / (question_id + ".json")
output.write_text(json_text(payload,indent=2),encoding="utf-8")
print(json_text(payload,indent=2))
print("Saved:",output)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{
  "model_key": "qwen35-9b",
  "business_id": "biz_101",
  "as_of_date": "2026-02-22",
  "question": "Hôm nay cửa hàng bán được bao nhiêu tiền?",
  "submission": {
    "id": "interactive_20261107_091946_278525",
    "type": "number",
    "answer": 13479900
  },
  "elapsed_seconds": 9.2033,
  "model_calls": 2,
  "trace": [
    {
      "step": 1,
      "raw": "{\"action\":\"plan\",\"measure\":\"revenue\",\"operation\":\"sum\",\"type\":\"number\",\"population\":\"sales\",\"evidence_id\":\"Q\",\"periods\":[{\"kind\":\"today\",\"date\":\"2026-02-22\"}],\"filters\":[{\"column\":\"status\",\"op\":\"eq\",\"value\":\"completed\",\"evidence_id\":\"Q\"}]}<|im_end|>",
      "preflight": {
        "availability": "available",
        "missing": [],
        "unsupported": [],
        "requirements": [],
        "coverage": {
          "start": "2025-06-01",
          "end": "2026-08-24",
          "declared_end": "2026-08-24"
        },
        "evidence_scope": "all_source_schemas_and_full_column_

Khi muốn chuyển sang fine-tune, dùng `DataMate_Colab_LoRA.ipynb` sau khi đã kiểm tra baseline.
Các checkpoint được khóa trong registry trước cutoff; giữ tổng tham số <=15B. Notebook này dùng train để thử, không dùng public/private test để tạo dữ liệu training.


Để thử Pinkstackorg, đặt `MODEL_KEY = "qwen35-upscaled14b-v2"` ở ô đầu rồi chạy lại từ đầu trong runtime mới. Tải/seal dùng revision đã pin riêng, không đổi model repo của key `qwen35-9b`. Xem `docs/qwen35-upscaled14b-v2-colab.md`. Khi đổi model trong batch, RUN_NAME có tiền tố model riêng; không resume vào thư mục Gemma.


## Khôi phục pipeline 6 — 08/11/2026
Theo yêu cầu rollback, source/loader đã trở về bản trước interface v7. Tải lại ZIP và notebook, restart kernel rồi chạy từ ô đầu. Ô load phải hiện Pipeline 6, backend 3.
Giữ mẫu 100/seed 20261103, RUN_LIMIT=None. Suffix mới random100-rollback-pipeline6-Nov8th-01 để không trộn/resume vào log v7 hoặc lượt v6 đã hoàn tất. Chọn Qwen/Gemma ở MODEL_KEY như trước.
MODEL_KEY, QUANTIZATION, AUTO_DISCONNECT và output lịch sử được giữ. Không có số đo tốc độ GPU mới sau rollback. Bản v7 và log đã lưu riêng trong outputs/rollback-pipeline6-2026-11-08/.


In [6]:
"""Copy this entire file into ONE cell below the loaded agent, then run it.

Diagnostic inference only. Reuses the current model; does not train or change
prompts/tools/decoding. The default draws uniformly from all training questions.
Reference labels never select that sample or enter the agent; scoring is later.
"""
from __future__ import annotations

from collections import Counter, defaultdict
from copy import deepcopy
import csv
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import random
import re
import statistics
import sys
import time
import unicodedata
import zipfile

from datamate.data.split import ensure_outside_dataset, read_jsonl
from datamate.domain import DataMateError, json_text
from datamate.debug_log import write_debug_log
from datamate.evaluation.scoring import score
from datamate.models.artifacts import sha256
from datamate.training.dataset import business_holdout, require_approved_source

DIAGNOSTIC_VERSION = 8

# CẤU HÌNH LƯỢT CHẠY: chỉ cần execute một lần sau ô load model.
SAMPLE_SIZE = 100
RUN_LIMIT = None              # Cùng mẫu 100 câu; đổi thành 7 nếu muốn smoke-test trước rồi resume.
SAMPLING_MODE = "random"
SAMPLING_SEED = 20261103      # Chỉ để lặp lại đúng mẫu; không đổi seed của model.
MAX_RUN_MINUTES = 90           # Trần mềm, gồm chuẩn bị doanh nghiệp; có thể resume phần chưa chạy.
AUTO_DISCONNECT = True        # True: thử đồng bộ Drive và ngắt Colab sau khi xuất ZIP.
RUN_NAME_SUFFIX = "random100-rollback-pipeline6-Nov8th-01"  # Giữ mẫu cũ, output mới cho pipeline 6.
KNOWN_CASE = None             # Không ép đưa câu lỗi cũ vào mẫu ngẫu nhiên.


def question_profile(text):
    original = unicodedata.normalize("NFC", text).lower()
    plain = "".join(ch for ch in unicodedata.normalize("NFD", original)
                    if not unicodedata.combining(ch)).replace("đ", "d")
    tokens = set(re.findall(r"\w+", plain))
    abbreviations = sorted(tokens & {"dthu", "dt", "bn", "bnhiu", "bnhiêu", "bnhieu", "nhiu",
                                    "dc", "sp", "sl", "hqua", "hnay", "trc", "vs", "hsd", "tb"})
    patterns = {
        "comparison": (2, r"cung ky|so voi|so sanh|chenh lech|tang hay giam|hon hay kem|ngay lien truoc|ngay ngay truoc"),
        "multi_level_aggregation": (3, r"(?:khach|san pham|mat hang|ngay).*(?:it nhat|tro len|dat .*dong|tu \d+ don)|co tu \d+"),
        "ratio": (1, r"(?:ti|ty) (?:le|trong)|phan tram|%"),
        "debt_or_cost": (2, r"lai (?:gop|rong)|gia von|tuoi no|tra no|con no|chua tra|ton kho|con ton"),
        "date_or_calendar": (1, r"\btet\b|am lich|thang chap|\bquy\b|thang.*lien nhau|\d+ ngay (?:ke tu|gan nhat)|ca (?:sang|chieu)"),
        "unit_conversion": (2, r"don vi (?:co ban|lon)|quy doi|nguyen.*le|bao nhieu.*(?:thung|hop).*bao nhieu"),
        "ranking_or_series": (1, r"\btop\s*\d|liet ke|moi thang|tung thang|nhieu nhat|cao nhat|ban chay"),
        "ranking_over_aggregates": (2, r"(?:ngay nao|thang nao).*(?:cao nhat|nhieu nhat)|(?:cao nhat|nhieu nhat).*vao (?:ngay|thang)"),
    }
    cues = [name for name, (_, pattern) in patterns.items() if re.search(pattern, plain)]
    score_hint = sum(patterns[name][0] for name in cues) + int(len(plain.split()) >= 22)
    return {"no_diacritics": original == plain and bool(re.search(r"[a-z]", plain)),
            "abbreviated": bool(abbreviations), "abbreviation_tokens": abbreviations,
            "complex_hint": score_hint >= 2, "complexity_score_hint": score_hint,
            "complexity_cues": cues}


def language_stress_plan(questions, reference_types, seed, sample_size, probe):
    development, holdout = business_holdout(questions, .2, seed)
    pool = sorted((q for q in questions if q.business_id in development), key=lambda q: q.id)
    if sample_size != 3 * len(development):
        raise DataMateError("language_stress_3 cần đúng 3 câu mỗi development business (72 câu với train hiện tại)")
    rng = random.Random(seed)
    rng.shuffle(pool)
    profiles = {q.id: question_profile(q.question) for q in pool}
    counts, type_counts, selected = Counter(), Counter(), []
    businesses = sorted(development)
    rng.shuffle(businesses)

    def take(candidates, prefer_abbreviation=False):
        if not candidates:
            raise DataMateError("Không đủ câu để chọn mẫu ngôn ngữ")
        q = min(candidates, key=lambda q: (
            -int(prefer_abbreviation and profiles[q.id]["abbreviated"]),
            type_counts[reference_types[q.id]], -profiles[q.id]["complexity_score_hint"]))
        selected.append(q)
        pool.remove(q)
        counts[q.business_id] += 1
        type_counts[reference_types[q.id]] += 1

    known = next((q for q in pool if probe == (q.business_id, q.as_of_date, q.question)), None)
    if known:
        take([known])
    for kind in ("no_diacritics", "complex_hint", "abbreviated"):
        for business in businesses:
            if counts[business] >= 3:
                continue
            existing = [q for q in selected if q.business_id == business]
            if any(profiles[q.id][kind] for q in existing):
                continue
            candidates = [q for q in pool if q.business_id == business and profiles[q.id][kind]]
            if not candidates:
                raise DataMateError(f"Không có câu thuộc nhóm {kind} cho {business}; cần xem lại tiêu chí, không tự bịa câu")
            take(candidates, prefer_abbreviation=kind == "no_diacritics")
    for business in businesses:
        while counts[business] < 3:
            take([q for q in pool if q.business_id == business])
    if len(selected) != sample_size or set(reference_types[q.id] for q in selected) != set(reference_types[q.id] for q in questions if q.business_id in development):
        raise DataMateError("Mẫu chưa đủ số câu/type; cần điều chỉnh chọn mẫu trước khi dùng GPU")
    for business in businesses:
        chosen = [profiles[q.id] for q in selected if q.business_id == business]
        if not all(any(profile[kind] for profile in chosen) for kind in ("no_diacritics", "abbreviated", "complex_hint")):
            raise DataMateError(f"Mẫu của {business} chưa phủ đủ nhóm ngôn ngữ/độ khó")
    return selected, sorted(development), sorted(holdout), known.id if known else None


def diagnostic_plan(questions, reference_types, seed=42, sample_size=120, probe=None, sampling_mode="balanced"):
    if sampling_mode == "random":
        if type(sample_size) is not int or not 1 <= sample_size <= len(questions):
            raise DataMateError("sample_size phải là số nguyên từ 1 đến tổng số câu train")
        pool = sorted(questions, key=lambda q: q.id)
        if len({q.id for q in pool}) != len(pool):
            raise DataMateError("Không chọn mẫu từ danh sách trùng question ID")
        # No reference types, text profiles, business quotas, forced cases,
        # previous-run exclusions or holdout filtering affect this draw.
        return random.Random(seed).sample(pool, sample_size), sorted({q.business_id for q in pool}), [], None
    if sampling_mode == "language_stress_3":
        return language_stress_plan(questions, reference_types, seed, sample_size, probe)
    if sampling_mode != "balanced":
        raise DataMateError("sampling_mode không hợp lệ")
    if sample_size < 1:
        raise DataMateError("sample_size phải dương")
    development, holdout = business_holdout(questions, 0.2, seed)
    pool = sorted((q for q in questions if q.business_id in development), key=lambda q: q.id)
    if any(q.id not in reference_types for q in pool):
        raise DataMateError("Train thiếu type tham chiếu khi chọn mẫu")
    target = min(sample_size, len(pool))
    if target < len(development):
        raise DataMateError("sample_size cần đủ ít nhất một câu mỗi development business")
    rng = random.Random(seed)
    businesses = sorted(development)
    rng.shuffle(businesses)
    rng.shuffle(pool)
    selected, type_counts, business_counts = [], Counter(), Counter()

    def take(candidates):
        question = min(candidates, key=lambda q: (type_counts[reference_types[q.id]], business_counts[q.business_id]))
        selected.append(question)
        pool.remove(question)
        type_counts[reference_types[question.id]] += 1
        business_counts[question.business_id] += 1

    known = next((q for q in questions if probe == (q.business_id, q.as_of_date, q.question)
                  and q.business_id in development), None)
    if known:
        take([known])
    for business in businesses:
        if not business_counts[business]:
            take([q for q in pool if q.business_id == business])
    while len(selected) < target:
        take(pool)
    return selected, sorted(development), sorted(holdout), known.id if known else None


def reference_type_index(path):
    # Only ID/type metadata are retained for stratification. Gold answers are
    # not supplied to the agent; values are used by write_report after inference.
    with Path(path).open(encoding="utf-8-sig") as handle:
        return {row["id"]: row["type"] for line in handle if line.strip()
                for row in [json.loads(line)]}


def runtime_snapshot(backend):
    import datamate
    result = {"python": sys.version, "backend": type(backend).__name__, "packages": {}}
    result['datamate_code_version']=datamate.__version__
    result['effective_generation']={'do_sample':False,'num_beams':1,'protocol_version':getattr(backend,'protocol_version',None)}
    result['generation_tokens'] = deepcopy(getattr(backend, 'generation_tokens', None))
    for package in ("torch", "transformers", "accelerate", "bitsandbytes", "datamate-agent"):
        try:
            result["packages"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result["packages"][package] = None
    torch = getattr(backend, "torch", None)
    if torch is not None:
        result["cuda_available"] = torch.cuda.is_available()
        result["cuda_version"] = torch.version.cuda
        if result["cuda_available"]:
            result["gpus"] = [{"name": torch.cuda.get_device_name(i),
                               "total_vram_gib": torch.cuda.get_device_properties(i).total_memory / 2**30}
                              for i in range(torch.cuda.device_count())]
    model = getattr(backend, "model", None)
    if model is not None:
        result["hf_device_map"] = {key: str(value) for key, value in getattr(model, "hf_device_map", {}).items()}
        result["parameter_devices"] = sorted({str(p.device) for p in model.parameters()})
        result["dtype"] = str(getattr(model, "dtype", "unknown"))
        generation = getattr(model, "generation_config", None)
        if generation is not None:
            result["generation_config"] = {name: getattr(generation, name, None)
                                           for name in ("do_sample", "num_beams", "eos_token_id", "pad_token_id")}
    tokenizer = getattr(backend, "tokenizer", None)
    if tokenizer is not None:
        result["tokenizer_class"] = type(tokenizer).__name__
        result['tokenizer_eos_token_id'] = getattr(tokenizer, 'eos_token_id', None)
        result['tokenizer_pad_token_id'] = getattr(tokenizer, 'pad_token_id', None)
        result["chat_template"] = getattr(tokenizer, "chat_template", None)
    return result


def source_hashes():
    import datamate
    root=Path(datamate.__file__).parent
    return {str(p.relative_to(root).with_suffix('')).replace('\\','.').replace('/','.'):sha256(p)
            for p in sorted(root.rglob('*.py'))}


class TimedBackend:
    def __init__(self, delegate):
        self.delegate, self.calls = delegate, []

    def __getattr__(self, name):
        return getattr(self.delegate, name)

    def generate(self, messages, remaining_seconds, **kwargs):
        started = time.monotonic()
        event = {"input_characters": sum(len(m.get("content", "")) for m in messages)}
        try:
            output = self.delegate.generate(messages, remaining_seconds, **kwargs)
            event["output_characters"] = len(output)
            return output
        except Exception as exc:
            event["error"] = f"{type(exc).__name__}: {exc}"
            raise
        finally:
            event["seconds"] = round(time.monotonic() - started, 4)
            generation=getattr(self.delegate,'last_generation',None)
            if generation:
                event['generation']=generation
            self.calls.append(event)


def diagnostic_flags(record, reference):
    flags = set()
    predicted = record["submission"]
    if predicted["type"] != "abstain" and predicted["type"] != reference["type"]:
        flags.add("type_mismatch")
    if predicted["type"] == "unanswerable" and reference["type"] != "unanswerable":
        flags.add("false_unanswerable")
    for event in record.get("trace", []):
        action, observation = event.get("action", {}), event.get("observation", {})
        error = observation.get("error") or event.get("error", "")
        if error:
            kind=observation.get('error_kind') or event.get('error_kind')
            if kind:
                flags.add(kind)
            if "Context vượt" in error:
                flags.add("context_limit")
            elif action.get("action") == "sql":
                flags.add("sql_error")
            elif "raw" in event and not action:
                flags.add("action_parse_error")
            else:
                flags.add("runtime_or_tool_error")
        if observation.get("truncated"):
            flags.add("truncated_sql_result")
        if event.get('loop_stopped'):
            flags.add('repeated_action_stopped')
    if predicted["type"] == "abstain":
        flags.add("abstain")
        if predicted.get("reason", "").startswith("Hết ngân sách"):
            flags.add("step_or_time_budget")
    return sorted(flags)


def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json_text(value, indent=2), encoding="utf-8")
    temporary.replace(path)


def write_report(destination, plan, manifest, target, runtime, batch_seconds):
    # Gold labels enter this reporting function only, after predictions exist.
    references = {r["id"]: r for r in read_jsonl(Path(manifest["settings"]["split_root"]) / "answers.jsonl")}
    records = [json.loads((destination / "logs" / (q.id + ".json")).read_text(encoding="utf-8"))
               for q in plan if (destination / "logs" / (q.id + ".json")).is_file()]
    for record in records:
        readable = destination / 'logs' / (record['id'] + '.md')
        if not readable.is_file():
            write_debug_log(readable, record)
    ids = {row["id"] for row in records}
    metrics = score([r["submission"] for r in records], list(references.values()), ids)
    outcomes = {row["id"]: row["result"] for row in metrics.pop("details")}
    review = [{**r, "reference": references[r["id"]], "outcome": outcomes[r["id"]],
               "question_profile": question_profile(r["question"]),
               "flags": diagnostic_flags(r, references[r["id"]])} for r in records]
    times = sorted(r["elapsed_seconds"] for r in records)
    groups = defaultdict(list)
    for row in records:
        groups[references[row["id"]]["type"]].append(row)
    by_type = {kind: score([r["submission"] for r in rows], list(references.values()), {r["id"] for r in rows})
               for kind, rows in groups.items()}
    for value in by_type.values():
        value.pop("details")
    by_language = {}
    for group in ("no_diacritics", "abbreviated", "complex_hint"):
        subset = [r for r in review if r["question_profile"][group]]
        value = score([r["submission"] for r in subset], list(references.values()), {r["id"] for r in subset})
        value.pop("details")
        by_language[group] = value
    summary = {**metrics, "diagnostic_only": True, "model_key": manifest["settings"]["model_key"],
               "planned_questions": len(plan), "requested_cumulative_limit": target,
               "completed_questions": len(records), "remaining_questions": len(plan) - len(records),
               "completed_businesses": len({r["business_id"] for r in records}),
               "status": "complete" if len(records) >= target else "partial",
               "planned_type_counts": manifest["planned_type_counts"],
               "planned_language_counts": manifest["planned_language_counts"],
               "by_question_group": by_language,
               "per_business_counts": dict(Counter(r["business_id"] for r in records)),
               "predicted_type_counts": dict(Counter(r["submission"]["type"] for r in records)),
               "mean_seconds": statistics.mean(times) if times else None,
               "median_seconds": statistics.median(times) if times else None,
               "p95_seconds": times[max(0, math.ceil(.95 * len(times)) - 1)] if times else None,
               "total_model_calls": sum(r["model_calls"] for r in records),
               "preparation_seconds": round(sum(e.get('preparation_elapsed_seconds',0) for r in records for e in r.get('trace',[])),4),
               "preparation_model_calls": sum(e.get('phase')=='preparation' and bool(e.get('model_call_attempted') or 'raw' in e) for r in records for e in r.get('trace',[])),
               "auto_projected_questions": sum(any(e.get('accepted') and 'auto_final' in e for e in r.get('trace',[])) for r in records),
               "schema_repair_attempts": sum(e.get('preparation_stage')=='targeted_schema_repair' and bool(e.get('model_call_attempted')) for r in records for e in r.get('trace',[])),
               "schema_repair_cache_hits": sum(e.get('preparation_stage')=='targeted_schema_repair' and bool(e.get('cache_hit')) for r in records for e in r.get('trace',[])),
               "flag_counts": dict(Counter(flag for r in review for flag in r["flags"])),
               "by_reference_type": by_type, "this_invocation_seconds": round(batch_seconds, 3),
               "model_load_excluded": True,
               "limitations": ("Mẫu ngẫu nhiên đều từ toàn bộ train, không lọc holdout hoặc câu cũ; không phải đánh giá trên tập chưa dùng để phát triển. " if manifest.get('sampling')=='random' else "Mẫu development có tiêu chí, không đại diện tỉ lệ câu toàn train. ")+
                              "PA chỉ tính câu đã chạy. Thời gian gồm chuẩn bị doanh nghiệp nhưng không gồm load model. Nhóm ngôn ngữ có thể chồng lấp."}
    save_json(destination / "summary.json", summary)
    save_json(destination / "runtime.json", runtime)
    (destination / "review.jsonl").write_text("".join(json_text(r) + "\n" for r in review), encoding="utf-8")
    (destination / "submission.jsonl").write_text("".join(json_text(r["submission"]) + "\n" for r in records), encoding="utf-8")
    with (destination / "review.csv").open("w", encoding="utf-8-sig", newline="") as handle:
        fields = ["id", "business_id", "question", "as_of_date", "outcome", "predicted_type", "reference_type",
                  "predicted_answer", "reference_answer", "elapsed_seconds", "model_calls", "flags",
                  "no_diacritics", "abbreviated", "abbreviation_tokens", "complex_hint", "complexity_cues"]
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for r in review:
            writer.writerow({**{key: r[key] for key in ("id", "business_id", "question", "as_of_date", "outcome", "elapsed_seconds", "model_calls")},
                             "predicted_type": r["submission"]["type"], "reference_type": r["reference"]["type"],
                             "predicted_answer": json_text(r["submission"].get("answer")),
                             "reference_answer": json_text(r["reference"].get("answer")), "flags": ",".join(r["flags"]),
                             **{key: r["question_profile"][key] for key in ("no_diacritics", "abbreviated", "complex_hint")},
                             "abbreviation_tokens": ",".join(r["question_profile"]["abbreviation_tokens"]),
                             "complexity_cues": ",".join(r["question_profile"]["complexity_cues"])})
    bundle = destination / "diagnostic_bundle.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(destination.rglob("*")):
            if path.is_file() and path != bundle and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(destination).as_posix())
    return summary, bundle


def disconnect_colab_after_report(bundle):
    """Optional, only called explicitly after a readable report is persisted."""
    with zipfile.ZipFile(bundle) as archive:
        if archive.testzip() is not None:
            raise DataMateError("ZIP chưa vượt kiểm tra; chưa ngắt Colab")
    import os
    if not os.environ.get("TBE_RUNTIME_ADDR"):
        print("Không có API quản lý runtime Colab. Hãy ngắt runtime GPU thủ công sau khi lấy REPORT.", flush=True)
        return
    try:
        from google.colab import drive, runtime
        print("Đang đồng bộ các ghi chờ trên Drive trước khi kết thúc runtime.", flush=True)
        drive.flush_and_unmount(timeout_ms=60000)
        print("Đã ghi REPORT; đang yêu cầu Colab kết thúc runtime để tiết kiệm Unit.", flush=True)
        runtime.unassign()
    except Exception as exc:
        print(f"Chưa xác nhận tự ngắt được runtime: {exc}. Hãy kiểm tra và ngắt thủ công.", flush=True)


def run_diagnostic(agent, output_dir, sources, limit=None, wall_budget_seconds=10800,
                   seed=42, sample_size=120, probe=None, sampling_mode="balanced"):
    if (limit is not None and limit < 1) or wall_budget_seconds <= 0:
        raise DataMateError("limit và wall_budget_seconds phải dương")
    if getattr(agent.backend, "counts_as_model", True) is False:
        raise DataMateError("Chẩn đoán này cần model thật; không chạy Demo")
    if getattr(agent,'pipeline_version',0)<6:
        raise DataMateError('Agent trong RAM là bản cũ. Thay source ZIP và khởi động lại kernel, rồi load pipeline >=6')
    if type(agent.backend).__name__=='TransformersBackend' and getattr(agent.backend,'protocol_version',0)<3:
        raise DataMateError('Backend trong RAM là bản cũ; khởi động lại kernel sau khi cập nhật source')
    original = agent.backend
    if getattr(original, "settings", agent.settings).to_dict() != agent.settings.to_dict():
        raise DataMateError("Config agent và backend không khớp; load lại model trước")
    split = agent.split
    approved = require_approved_source(split.root, sources)
    reference_types = {} if sampling_mode == "random" else reference_type_index(split.root / "answers.jsonl")
    plan, development, holdout, known = diagnostic_plan(split.questions(), reference_types, seed, sample_size, probe, sampling_mode)
    if sampling_mode == "random":
        # Read ID/type metadata for reporting only AFTER the sample is fixed.
        reference_types = reference_type_index(split.root / "answers.jsonl")
    settings = {**agent.settings.to_dict(), "split_root": str(split.root.resolve())}
    model_manifest = Path(settings["model_dir"]) / "artifact_manifest.json"
    if not model_manifest.is_file():
        raise DataMateError("Không thấy manifest của model đang dùng; kiểm tra model_dir")
    identity = {"diagnostic_version": DIAGNOSTIC_VERSION, "pipeline_version":agent.pipeline_version, "settings": settings,
                "source_fingerprint": approved["fingerprint"], "source_hashes": source_hashes(),
                "model_manifest_sha256": sha256(model_manifest), "seed": seed,
                "development_businesses": development if sampling_mode != "random" else None,
                "reserved_holdout_businesses": holdout,
                "sampling_population_businesses": development,
                "sampling_scope": "all_training_questions" if sampling_mode == "random" else "development_businesses",
                "sampling": sampling_mode,
                "planned_type_counts": dict(Counter(reference_types[q.id] for q in plan)),
                "planned_language_counts": {key: sum(question_profile(q.question)[key] for q in plan)
                                            for key in ("no_diacritics", "abbreviated", "complex_hint")},
                "known_case_id": known, "question_ids": [q.id for q in plan]}
    digest = hashlib.sha256(json_text(identity, sort_keys=True).encode()).hexdigest()
    destination = ensure_outside_dataset(output_dir, split.root)
    destination.mkdir(parents=True, exist_ok=True)
    manifest_path = destination / "run_manifest.json"
    if manifest_path.exists():
        if json.loads(manifest_path.read_text(encoding="utf-8"))["identity_hash"] != digest:
            raise DataMateError("Lượt cũ khác model/config/code/data/sample. Đổi RUN_NAME; không trộn benchmark.")
    elif any(destination.iterdir()):
        raise DataMateError("Output có file nhưng thiếu manifest; dùng RUN_NAME mới")
    manifest = {**identity, "identity_hash": digest}
    save_json(manifest_path, manifest)
    (destination / "logs").mkdir(exist_ok=True)
    (destination / "selected_questions.jsonl").write_text(
        "".join(json_text({"id": q.id, "business_id": q.business_id, **q.prompt_dict(),
                          "question_profile": question_profile(q.question)}) + "\n" for q in plan), encoding="utf-8")
    runtime = runtime_snapshot(original)
    print("Model:", settings["model_key"], "quantization:", settings["quantization"], flush=True)
    print("GPU/devices:", json_text({k: v for k, v in runtime.items() if k != "chat_template"}), flush=True)
    target = len(plan) if limit is None else min(limit, len(plan))
    print(f"Plan: {len(plan)} câu; lượt này chạy đến câu {target}.", flush=True)
    if sampling_mode == 'random':
        print(f"Ngẫu nhiên đều từ toàn bộ train; seed={seed}. Không cân đối type/doanh nghiệp/ngôn ngữ, không lọc holdout hoặc câu cũ.", flush=True)
    else:
        print(f"Giữ riêng {len(holdout)} doanh nghiệp holdout.", flush=True)
    print("Loại đáp án:", json_text(identity["planned_type_counts"]), flush=True)
    print("Nhóm câu hỏi (có thể chồng lấp):", json_text(identity["planned_language_counts"]), flush=True)
    print("Giữ nguyên prompt, SQL tools, decoding và ngân sách của agent hiện tại.", flush=True)
    timer, started = TimedBackend(original), time.monotonic()
    try:
        agent.backend = timer
        for index, question in enumerate(plan[:target], 1):
            path = destination / "logs" / (question.id + ".json")
            if path.exists():
                print(f"[{index}/{target}] đã có {question.id}, bỏ qua", flush=True)
                continue
            remaining = wall_budget_seconds - (time.monotonic() - started)
            if remaining <= 0:
                print("Hết ngân sách lượt chạy; có thể chạy lại để tiếp tục.", flush=True)
                break
            timer.calls = []
            print(f"[{index}/{target}] {question.business_id} | {question.as_of_date} | {question.question}", flush=True)
            result = agent.run(question, remaining_seconds=remaining)
            record = {"id": question.id, "business_id": question.business_id, **question.prompt_dict(),
                      "rulebook_sha256":split.rulebook_hash,
                      **result.to_dict(question.id), "generation_calls": list(timer.calls)}
            save_json(path, record)
            write_debug_log(path.with_suffix('.md'), record)
            print(f"  -> {result.answer.type} | {result.elapsed_seconds:.1f}s | {result.model_calls} model calls", flush=True)
    finally:
        agent.backend = original
        summary, bundle = write_report(destination, plan, manifest, target, runtime, time.monotonic() - started)
        print(json_text(summary, indent=2), flush=True)
        print("REPORT:", bundle, flush=True)
    return summary, bundle


if __name__ == "__main__":
    # Copy the ENTIRE file to the current notebook, below its model-loading cell.
    # Do not run it in a new kernel: it reuses the existing `agent`.
    if any(name not in globals() for name in ("agent", "WORK", "SAVE")):
        raise RuntimeError("Dán toàn bộ file vào ô mới trong notebook inference đã load model (cần agent, WORK, SAVE).")
    RUN_NAME = agent.settings.model_key + "-" + RUN_NAME_SUFFIX
    diagnostic_summary, diagnostic_bundle = run_diagnostic(
        agent, Path(SAVE) / "diagnostics" / RUN_NAME,
        sources=Path(WORK) / "configs/training_sources.json",
        limit=RUN_LIMIT,
        sample_size=SAMPLE_SIZE, wall_budget_seconds=MAX_RUN_MINUTES * 60, probe=KNOWN_CASE,
        sampling_mode=SAMPLING_MODE, seed=SAMPLING_SEED,
    )
    if AUTO_DISCONNECT:
        disconnect_colab_after_report(diagnostic_bundle)


Model: qwen35-9b quantization: none
GPU/devices: {"python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]", "backend": "TransformersBackend", "packages": {"torch": "2.14.1+cu130", "transformers": "5.22.1", "accelerate": "1.15.0", "bitsandbytes": "0.50.2", "datamate-agent": "0.2.0"}, "datamate_code_version": "0.2.0", "effective_generation": {"do_sample": false, "num_beams": 1, "protocol_version": 3}, "generation_tokens": {"checkpoint_eos_token_ids": [248044], "tokenizer_eos_token_id": 248046, "tokenizer_pad_token_id": 248044, "chat_turn_token_ids": {"<|im_end|>": 248046}, "effective_eos_token_ids": [248044, 248046], "effective_pad_token_id": 248044}, "cuda_available": true, "cuda_version": "13.0", "gpus": [{"name": "NVIDIA RTX PRO 6000 Blackwell Server Edition", "total_vram_gib": 94.97076416015625}], "hf_device_map": {}, "parameter_devices": ["cuda:0"], "dtype": "torch.bfloat16", "generation_config": {"do_sample": null, "num_beams": null, "eos_token_id": 248044, "pad_token_id": nu